# 实验 03：正则表达式与网络结构化数据

`20244080329_吴卓斯_exp03.ipynb`

本实验分三块：先用 `re` 做基础匹配，再做分组、回溯引用和替换，最后用 BeautifulSoup 解析一段静态 HTML。
环境是单独的 conda 环境 `exp03`，不是 base。

## 0. 身份与环境记录

In [1]:
import hashlib
import os
import platform
import uuid
import sys

# 修改为自己的学号
STUDENT_ID = 20244080329

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"

print(f"[学号]       {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")
print(f"[Python]   {sys.version.split()[0]}")

[Conda 环境] exp03
[学号]       20244080329
[机器指纹] 72a4fda4ca0e6d59
[RUN_ID]   20244080329-72a4fda4ca0e6d59
[Python]   3.11.17


## 实验 1：正则表达式的基础匹配

先说结论：`re.findall` 返回的是**所有不重叠匹配**，有捕获组时返回组的内容，没有捕获组时返回整段匹配文本。

In [2]:
import re

text = "cat on mat; later: cats and dogs"

print("原文 :", text)
print(r"\d+      ->", re.findall(r"\d+", text))
print(r"\s+      ->", re.findall(r"\s+", text))
print(r"\bcat\b  ->", re.findall(r"\bcat\b", text))

原文 : cat on mat; later: cats and dogs
\d+      -> []
\s+      -> [' ', ' ', ' ', ' ', ' ', ' ']
\bcat\b  -> ['cat']


`\d+` 一个数字都没有，所以是空列表；`\s+` 把三处空白都抓出来了（空格、分号后的空格、冒号后的空格），返回的是白字符串本身，看起来像空的；`\bcat\b` 只匹配独立的 `cat`，
所以后面的 `cats` 不会被算进来——`\b` 是单词边界，`cat` 后面紧跟 `s` 不算边界。

In [3]:
s = "cat dog bird CAT"

print("原文          :", s)
print(r"cat|dog       ->", re.findall(r"cat|dog", s))
print(r"cat|dog (?i)  ->", re.findall(r"cat|dog", s, flags=re.I))
print()

t = "aaaa"
print(r"a|aa|aaa 在 'aaaa' 上 ->", re.findall(r"a|aa|aaa", t))
print(r"aaa|aa|a 在 'aaaa' 上 ->", re.findall(r"aaa|aa|a", t))

原文          : cat dog bird CAT
cat|dog       -> ['cat', 'dog']
cat|dog (?i)  -> ['cat', 'dog', 'CAT']

a|aa|aaa 在 'aaaa' 上 -> ['a', 'a', 'a', 'a']
aaa|aa|a 在 'aaaa' 上 -> ['aaa', 'a']


`cat|dog` 默认区分大小写，所以大写 `CAT` 匹配不到；加 `re.I` 就变成三条。

第二个对比是重点：**交替分支是从左往右依次尝试的**。写 `a|aa|aaa` 时，第一个分支 `a` 就把每个字符吃掉了，得到四个 `'a'`；
写 `aaa|aa|a` 时，引擎先试最长的 `aaa`，剩下一个 `a` 再匹配，就变成 `['aaa', 'a']`。所以想让长匹配优先，就得把长分支写前面。

In [4]:
q = "ct cat caat caaat"
print("原文:", q)
for p in [r"ca*t", r"ca+t", r"ca?t", r"ca{2}t", r"ca{1,2}t"]:
    print(f"{p:10} -> {re.findall(p, q)}")

原文: ct cat caat caaat
ca*t       -> ['ct', 'cat', 'caat', 'caaat']
ca+t       -> ['cat', 'caat', 'caaat']
ca?t       -> ['ct', 'cat']
ca{2}t     -> ['caat']
ca{1,2}t   -> ['cat', 'caat']


- `*` 是 0 到多次，`a` 可以没有，所以 `ct` 也算匹配；
- `+` 是 1 到多次，`ct` 被排除；
- `?` 是 0 或 1 次，只能匹配到 `ct` 和 `cat`；
- `{2}` 是固定两次，只有 `caat`；
- `{1,2}` 是 1 到 2 次，`cat` 和 `caat` 都算。

顺带注意，`ca*t` 也能在 `caaat` 里匹配到子串，因为正则是"找子串"而不是"整串相等"。

## 实验 2：分组、回溯引用与替换

捕获组用圆括号，`findall` 在有多个组时返回元组；`sub` 的替换串里可以用 `\1`、`\2` 引用组，
命名组则用 `(?P<name>...)` 配 `(?P=name)`。

In [5]:
mails = "联系：alice@example.com；备选：bob@test.org"
pat = re.compile(r"([\w.+-]+)@([\w-]+(?:\.[\w-]+)+)")

print("findall:", pat.findall(mails))
print()
for m in pat.finditer(mails):
    print(f"完整匹配 {m.group(0):22} 用户名 {m.group(1):8} 域名 {m.group(2)}")

findall: [('alice', 'example.com'), ('bob', 'test.org')]

完整匹配 alice@example.com      用户名 alice    域名 example.com
完整匹配 bob@test.org           用户名 bob      域名 test.org


In [6]:
words = "cat cat, dog cat, bird bird"
dup = re.findall(r"\b(\w+)\s+\1\b", words)
print("原文:", words)
print("连续重复的词:", dup)
print("替换掉重复:", re.sub(r"\b(\w+)\s+\1\b", r"\1", words))

原文: cat cat, dog cat, bird bird
连续重复的词: ['cat', 'bird']
替换掉重复: cat, dog cat, bird


`(\w+)` 记住一个词，`\s+` 允许中间有空白，`\1` 要求后面再来一次**同样的**内容。
所以 `cat cat` 和 `bird bird` 命中，`dog cat` 不命中。这个技巧在清理重复录入的文本时很好用。

In [7]:
dates = "2024-05-01\n2024/05/02\n2024.05.03"
pat = re.compile(r"(\d{4})([-/.])(\d{2})\2(\d{2})")

converted = pat.sub(lambda m: f"{m.group(3)}-{m.group(4)}-{m.group(1)}", dates)

print("转换前:")
print(dates)
print("\n转换后:")
print(converted)
print("\n行数:", len(dates.splitlines()), "->", len(converted.splitlines()))

转换前:
2024-05-01
2024/05/02
2024.05.03

转换后:
05-01-2024
05-02-2024
05-03-2024

行数: 3 -> 3


三种分隔符用 `[-/.]` 一个字符类解决，再用 `\2` 引用它，保证前后分隔符一致（不会出现 `2024-05/01` 这种混搭）。
替换用函数而不是 `\3-\4-\1` 字符串，是为了顺手统一成短横线；如果要求保留原分隔符，写成 `r"\3\2\4\2\1"` 就行。

In [8]:
phones = "联系我：123-456-7890 或 987.654.3210"
pat = re.compile(r"(\d{3})[.-](\d{3})[.-](\d{4})")

print("原文:", phones)
print("统一:", pat.sub(r"(\1) \2-\3", phones))

原文: 联系我：123-456-7890 或 987.654.3210
统一: 联系我：(123) 456-7890 或 (987) 654-3210


## 实验 3：用 BeautifulSoup 解析 HTML

这段 HTML 是写死在字符串里的，不涉及下载。`html.parser` 是标准库自带的解析器，够用且不用装额外依赖。

In [9]:
html_doc = """
<html><head><title>实验课程</title></head><body>
<h1>数据分析</h1>
<p class="course">第六讲：正则表达式</p>
<p class="course">第七讲：结构化数据</p>
<a href="https://example.org/lecture06">讲义六</a>
<a href="https://example.org/lecture07">讲义七</a>
</body></html>
"""

from bs4 import BeautifulSoup

soup = BeautifulSoup(html_doc, "html.parser")

print("title 文本 :", soup.title.get_text())
print("h1 标签名  :", soup.h1.name)
print("h1 文本    :", soup.h1.get_text())

title 文本 : 实验课程
h1 标签名  : h1
h1 文本    : 数据分析


In [10]:
print("class=course 的段落：")
for i, p in enumerate(soup.find_all("p", class_="course"), 1):
    print(f"  {i}. {p.get_text()}")

print("\n所有链接：")
for a in soup.find_all("a"):
    print(f"  文字={a.get_text():6} href={a['href']}")

class=course 的段落：
  1. 第六讲：正则表达式
  2. 第七讲：结构化数据

所有链接：
  文字=讲义六    href=https://example.org/lecture06
  文字=讲义七    href=https://example.org/lecture07


In [11]:
whole = soup.get_text()
para = soup.find("p", class_="course").get_text()
link = soup.find("a").get_text()

print("整个文档 get_text() 的原始表示：")
print(repr(whole))
print()
print("整页文本（按行去空行后）：")
print("\n".join([ln.strip() for ln in whole.splitlines() if ln.strip()]))
print()
print("单个段落:", repr(para))
print("单个链接:", repr(link))

整个文档 get_text() 的原始表示：
'\n实验课程\n数据分析\n第六讲：正则表达式\n第七讲：结构化数据\n讲义六\n讲义七\n\n'

整页文本（按行去空行后）：
实验课程
数据分析
第六讲：正则表达式
第七讲：结构化数据
讲义六
讲义七

单个段落: '第六讲：正则表达式'
单个链接: '讲义六'


三者的区别很清楚：`soup.get_text()` 把整棵树里所有文本按文档顺序拼起来，中间保留原有的换行和缩进；
段落和链接的 `get_text()` 只是取出自身包含的那一段文字，没有多余空白。

如果只想拿干净文本，可以用 `get_text(strip=True)` 或者 `separator=" "` 控制拼接方式——
不加参数时，标签之间原本的换行会原样留下，打印出来常常是一堆空行，这也是新手最容易困惑的地方。

## 小结

正则部分记住三件事：`findall` 找的是**所有不重叠子串**、交替分支**从左往右**尝试、`{m,n}` 这类量词只作用于紧挨着它的那一个字符。
解析部分则是"先建树再取值"：`BeautifulSoup(...)` 负责把字符串变成可查询的树，`find/find_all` 负责定位节点，`get_text()` 负责把节点里的文字拿出来。